# Auto-Slicing Dimensions

Any dimension that isn't `time_dim` or `column_dim` is automatically sliced —
tsam_xarray runs an independent aggregation for each coordinate and concatenates the results.

In [ ]:
import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

da = sample_energy_data(n_days=30)
da.to_dataframe()

## column_dims clusters together, the rest is auto-sliced

When we set `column_dims=["variable", "region"]`, the remaining `scenario` dimension is automatically sliced — each scenario gets its own independent clustering.

In [ ]:
result = tsam_xarray.aggregate(da, n_clusters=4, column_dims=["variable", "region"])
result.typical_periods.to_dataframe("energy")

The `scenario` dimension appears on all results. Each scenario has its own cluster assignments — cluster 0 in "low" is unrelated to cluster 0 in "high".

In [ ]:
result.cluster_weights.to_dataframe("cluster weights")

In [ ]:
result.accuracy.rmse.to_dataframe("RMSE")

## Multiple slice dims

Multiple extra dims are all sliced independently. Here, both `region` and `scenario` are sliced — one aggregation per `(region, scenario)` combination.

In [ ]:
# Only variable is column dim — region and scenario are both auto-sliced
result_multi = tsam_xarray.aggregate(da, n_clusters=4, column_dims="variable")
print("Result dims:", result_multi.typical_periods.dims)
result_multi.typical_periods.to_dataframe("energy").head(10)

Note that the stacked MultiIndex is **automatically unstacked** in the results — `variable` and `region` appear as separate dimensions. No manual `.unstack()` needed.

In [ ]:
result.typical_periods.sel(scenario="low", variable="solar").plotly.imshow(
    x="timestep", y="cluster", facet_col="region", text_auto=".2f"
)